# Lesson 05: Training and Sampling Strategies

Practical training details for discrete diffusion: LR schedules, loss weighting, advanced sampling, and D3PM vs MDLM comparison.

In [ ]:
%pip install torch numpy matplotlib datasets tqdm --quiet

In [ ]:
import sys
sys.path.insert(0, '../../..')
sys.path.insert(0, '../lesson03-d3pm-from-scratch')
sys.path.insert(0, '../lesson04-mdlm')

import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from shared.utils.seed import set_seed
from shared.utils.device import get_device
from shared.datasets.text import SimpleTokenizer, TextDataset

set_seed(42)
device = get_device()
print(f"Using device: {device}")

## 1. Loss Weighting Strategies

Different timesteps contribute differently to training. Let's visualize the weighting strategies.

In [ ]:
from src.training_utils import importance_weight_timesteps

T = 100
t = torch.arange(1, T + 1)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, strategy in zip(axes, ['uniform', 'snr', 'truncated']):
    weights = importance_weight_timesteps(t, T, strategy)
    ax.plot(t.numpy(), weights.numpy())
    ax.set_title(f'{strategy} weighting')
    ax.set_xlabel('Timestep t')
    ax.set_ylabel('Weight')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2. Learning Rate Schedules

In [ ]:
from src.training_utils import get_cosine_schedule_with_warmup

# Simulate a scheduler
dummy_model = torch.nn.Linear(10, 10)
optimizer = torch.optim.Adam(dummy_model.parameters(), lr=3e-4)
scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=100, num_training_steps=2000)

lrs = []
for step in range(2000):
    lrs.append(optimizer.param_groups[0]['lr'])
    optimizer.step()
    scheduler.step()

plt.figure(figsize=(8, 4))
plt.plot(lrs)
plt.xlabel('Training Step')
plt.ylabel('Learning Rate')
plt.title('Cosine Schedule with Linear Warmup')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Prepare Data and Train Both Models

In [ ]:
# Shared training data
texts = [
    "the cat sat on the mat",
    "a dog ran in the park",
    "the sun is bright today",
    "birds fly in the sky",
    "fish swim in the sea",
    "the moon shines at night",
    "rain falls from the clouds",
    "flowers grow in the garden",
    "children play in the yard",
    "the wind blows through trees",
] * 100

tokenizer = SimpleTokenizer(texts, level="char")
dataset = TextDataset(texts, tokenizer, seq_len=32)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

print(f"Vocab: {tokenizer.vocab_size}, Dataset: {len(dataset)}")

In [ ]:
# Train D3PM
from src.d3pm_model import D3PMDenoiser, D3PM

d3pm_denoiser = D3PMDenoiser(
    vocab_size=tokenizer.vocab_size, d_model=64, n_heads=4,
    n_layers=3, max_seq_len=32, dropout=0.1,
).to(device)

d3pm = D3PM(
    denoiser=d3pm_denoiser, vocab_size=tokenizer.vocab_size,
    num_timesteps=100, schedule="absorbing",
    mask_token_id=tokenizer.mask_id, device=device,
)

optimizer_d3pm = torch.optim.Adam(d3pm_denoiser.parameters(), lr=3e-4)

d3pm_losses = []
for epoch in range(15):
    total = 0.0
    n = 0
    for batch in dataloader:
        batch = batch.to(device)
        loss = d3pm.train_loss(batch)
        optimizer_d3pm.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(d3pm_denoiser.parameters(), 1.0)
        optimizer_d3pm.step()
        total += loss.item()
        n += 1
    d3pm_losses.append(total / n)
    if (epoch + 1) % 5 == 0:
        print(f"D3PM Epoch {epoch+1}: loss={d3pm_losses[-1]:.4f}")

In [ ]:
# Train MDLM
from src.mdlm import MDLMDenoiser, MDLM

mdlm_denoiser = MDLMDenoiser(
    vocab_size=tokenizer.vocab_size, d_model=64, n_heads=4,
    n_layers=3, max_seq_len=32, dropout=0.1,
).to(device)

mdlm = MDLM(
    denoiser=mdlm_denoiser, vocab_size=tokenizer.vocab_size,
    mask_token_id=tokenizer.mask_id, num_timesteps=100,
    schedule_type="cosine", device=device,
)

optimizer_mdlm = torch.optim.Adam(mdlm_denoiser.parameters(), lr=3e-4)

mdlm_losses = []
for epoch in range(15):
    total = 0.0
    n = 0
    for batch in dataloader:
        batch = batch.to(device)
        loss = mdlm.train_loss(batch)
        optimizer_mdlm.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(mdlm_denoiser.parameters(), 1.0)
        optimizer_mdlm.step()
        total += loss.item()
        n += 1
    mdlm_losses.append(total / n)
    if (epoch + 1) % 5 == 0:
        print(f"MDLM Epoch {epoch+1}: loss={mdlm_losses[-1]:.4f}")

In [ ]:
# Compare training curves
plt.figure(figsize=(8, 4))
plt.plot(d3pm_losses, label='D3PM')
plt.plot(mdlm_losses, label='MDLM')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('D3PM vs MDLM Training Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Compare Samples

In [ ]:
from src.compare_models import print_comparison

# Generate samples from both models
d3pm_tokens = d3pm.sample(batch_size=10, seq_len=32, temperature=0.8)
mdlm_tokens = mdlm.sample(batch_size=10, seq_len=32, temperature=0.8)

d3pm_texts = [tokenizer.decode(s.cpu().tolist()) for s in d3pm_tokens]
mdlm_texts = [tokenizer.decode(s.cpu().tolist()) for s in mdlm_tokens]

reference = list(set(texts))  # unique training texts
print_comparison(d3pm_texts, mdlm_texts, reference)

## 5. Advanced Sampling: Top-k and Top-p

In [ ]:
from src.training_utils import sample_with_temperature

# Demonstrate different sampling strategies using the MDLM model
strategies = [
    {'temperature': 0.5, 'top_k': 0, 'top_p': 1.0, 'name': 'temp=0.5'},
    {'temperature': 1.0, 'top_k': 10, 'top_p': 1.0, 'name': 'top_k=10'},
    {'temperature': 1.0, 'top_k': 0, 'top_p': 0.9, 'name': 'top_p=0.9'},
    {'temperature': 0.8, 'top_k': 20, 'top_p': 0.95, 'name': 'combined'},
]

for strat in strategies:
    print(f"\nStrategy: {strat['name']}")
    # Simple demo: mask and predict
    x_0 = dataset[0].unsqueeze(0).to(device)
    mask = torch.rand_like(x_0.float()) < 0.5
    x_t = x_0.clone()
    x_t[mask] = tokenizer.mask_id
    
    t = torch.tensor([0.5], device=device)
    with torch.no_grad():
        logits = mdlm_denoiser(x_t, t)
    
    tokens = sample_with_temperature(
        logits[0], temperature=strat['temperature'],
        top_k=strat['top_k'], top_p=strat['top_p'],
    )
    # Fill in masked positions
    result = x_0[0].clone()
    result[mask[0]] = tokens[mask[0]]
    print(f"  Original:  '{tokenizer.decode(x_0[0].cpu().tolist())}'")
    print(f"  Masked:    '{tokenizer.decode(x_t[0].cpu().tolist())}'")
    print(f"  Filled:    '{tokenizer.decode(result.cpu().tolist())}'")

## 6. Perplexity Proxy Evaluation

In [ ]:
from src.training_utils import compute_perplexity_proxy

# Evaluate both models on the training data
eval_batch = next(iter(dataloader)).to(device)

d3pm_ppl = compute_perplexity_proxy(
    d3pm_denoiser, eval_batch, tokenizer.mask_id, device=device
)
mdlm_ppl = compute_perplexity_proxy(
    mdlm_denoiser, eval_batch, tokenizer.mask_id, device=device
)

print(f"Perplexity proxy (lower is better):")
print(f"  D3PM: {d3pm_ppl:.4f}")
print(f"  MDLM: {mdlm_ppl:.4f}")

## Key Takeaways

1. **Loss weighting** can focus training on the most informative timesteps.
2. **Cosine LR with warmup** is the standard schedule for stable training.
3. **Top-k and top-p filtering** improve sample quality beyond temperature alone.
4. **MDLM** is generally simpler to train and tune compared to D3PM.

## What's Next

Apply everything you have learned in the **Lab: Train a Discrete Diffusion Model** on TinyStories!